# 실습 01 · Colab & LLM 코딩 워밍업
### 제약바이오 AI 기초 (2026)

**이 노트북에서 배우는 것**
- Google Colab 이 무엇이고 왜 쓰는지 (설치 0, 무료 GPU)
- 코드 셀 실행, 라이브러리 설치, 파일 업로드
- pandas 로 제약 데이터를 처음 열어보기
- **LLM(Gemini/Claude)에게 코드를 시키는 법** — 이 강의의 핵심 작업방식

> 💡 이 과정은 "파이썬 문법을 외우는 강의"가 아닙니다.
> **"현장 문제를 정의 → LLM에게 코드를 요청 → Colab에서 실행·검증"** 하는
> 실무 워크플로우를 몸에 익히는 것이 목표입니다.


## 1. Colab 이란?
- 브라우저만 있으면 되는 **무료 파이썬 실행 환경** (구글 계정 필요)
- 내 노트북 성능과 무관하게 구글 서버에서 실행됨 → **무료 GPU/TPU** 사용 가능
- 라이브러리가 대부분 미리 깔려 있고, 없으면 `!pip install` 한 줄로 설치

**셀 실행 방법**: 셀을 클릭 후 `Shift + Enter`  (또는 왼쪽 ▶ 버튼)


In [ ]:
# [실행해보기] 셀을 클릭하고 Shift+Enter 를 눌러보세요.
print("안녕하세요, 제약바이오 AI 과정에 오신 것을 환영합니다!")
1 + 2 * 3   # 마지막 줄의 결과는 자동으로 출력됩니다

In [ ]:
# 지금 이 Colab 에 어떤 하드웨어가 붙어있는지 확인
import sys, platform
print("파이썬 버전 :", sys.version.split()[0])
print("운영체제    :", platform.platform())

# GPU 가 붙어있는지 확인 (상단 메뉴 [런타임]>[런타임 유형 변경]에서 GPU 선택 가능)
!nvidia-smi -L 2>/dev/null || echo "현재 GPU 미할당 (CPU 런타임)" 

## 2. 라이브러리 설치와 불러오기
파이썬은 **도구상자(라이브러리)** 를 불러와 쓰는 언어입니다.
- `pandas` : 엑셀 같은 표(데이터프레임) 다루기
- `numpy`  : 숫자 계산
- `matplotlib` : 그래프

이미 설치돼 있으므로 `import` 만 하면 됩니다. (없다면 `!pip install 이름`)


In [ ]:
import pandas as pd          # 표 데이터
import numpy as np             # 숫자 계산
import matplotlib.pyplot as plt  # 그래프
print("라이브러리 준비 완료 ✅")

## 3. 제약 데이터 처음 열어보기
실제 신약개발에서 가장 유명한 공개 데이터 중 하나인 **용해도(solubility) 데이터**를
인터넷에서 바로 불러오겠습니다. (엑셀을 여는 것과 똑같습니다)

- 각 행 = 하나의 화합물(분자)
- `smiles` : 분자를 글자로 표현한 것 (예: 아스피린 = `CC(=O)Oc1ccccc1C(=O)O`)
- `measured log solubility ...` : 실제로 측정한 용해도 값 (신약이 되려면 물에 잘 녹아야 함)


In [ ]:
# 인터넷 CSV 파일을 pandas 로 바로 읽기 (Delaney/ESOL 용해도 데이터, 1128개 분자)
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df = pd.read_csv(url)

print("데이터 크기 (행, 열):", df.shape)
df.head()   # 앞 5줄 미리보기

In [ ]:
# 어떤 열(항목)들이 있는지, 숫자 열의 요약 통계
print("== 열 목록 ==")
print(list(df.columns), "\n")
df.describe()   # 평균/최소/최대 등 자동 요약

In [ ]:
# 용해도 값의 분포를 히스토그램으로 (현장에서 '데이터를 눈으로 보는' 첫 단계)
col = "measured log solubility in mols per litre"
plt.figure(figsize=(7,4))
plt.hist(df[col], bins=40, color="#4C72B0", edgecolor="white")
plt.xlabel("측정 용해도 (log mol/L)"); plt.ylabel("화합물 수")
plt.title("공개 신약 데이터의 용해도 분포")
plt.show()

## 4. ⭐ 핵심 워크플로우: LLM에게 코드를 시키기
파이썬 문법을 몰라도 됩니다. **원하는 것을 정확히 말로 설명**하면 Gemini/Claude 가 코드를 써 줍니다.

### 좋은 프롬프트의 4가지 요소
1. **역할/맥락**: "너는 파이썬 데이터 분석가야. 나는 파이썬 초보야."
2. **데이터 설명**: "df 라는 데이터프레임이 있고, 열은 smiles, 용해도 값이 있어."
3. **하고 싶은 것(구체적으로)**: "용해도가 가장 낮은 화합물 10개를 표로 보여줘."
4. **형식/제약**: "설명은 한글 주석으로, matplotlib 로 그래프도 그려줘."

### 예시 프롬프트 (그대로 복사해서 Gemini 에 붙여보세요)
```
너는 친절한 파이썬 데이터 분석 튜터야. 나는 파이썬을 처음 배우는 제약 연구원이야.
pandas 데이터프레임 df 가 있고, 열은 'Compound ID', 'smiles',
'measured log solubility in mols per litre' 야.
용해도가 가장 낮은(=물에 안 녹는) 화합물 상위 10개를 골라서
Compound ID 와 용해도만 표로 출력하는 코드를 써줘. 코드에는 한글 주석을 달아줘.
```
아래는 그렇게 받은 코드의 예시입니다 👇


In [ ]:
# 👇 LLM 이 만들어줄 법한 코드 (직접 실행해보며 결과를 검증합니다)
col = "measured log solubility in mols per litre"

# 용해도가 낮은 순(오름차순)으로 정렬 후 상위 10개
worst10 = df.sort_values(col, ascending=True).head(10)

# 보고 싶은 열만 선택해서 출력
worst10[["Compound ID", col]]

### 🔁 실습: 여러분 차례입니다
아래 빈 셀에서, LLM 에게 다음을 요청해 코드를 받아 붙여넣고 실행해 보세요.
- "용해도가 **가장 높은**(물에 잘 녹는) 화합물 5개를 보여줘"
- "smiles 글자 길이를 새로운 열 `len` 으로 추가하고, 분자 크기와 용해도의 관계를 산점도로 그려줘"

> **검증 습관**: LLM 코드는 항상 실행해서 결과가 말이 되는지 확인하세요.
> 에러가 나면 에러 메시지를 그대로 LLM 에 붙여넣고 "이 에러 고쳐줘" 라고 하면 됩니다.


In [ ]:
# ✏️ 여기에 LLM 에게 받은 코드를 붙여넣고 실행하세요


## 정리
- Colab = 설치 없이 브라우저에서 파이썬 실행 (무료 GPU)
- pandas 로 CSV 를 열고 `.head()`, `.describe()`, 그래프로 데이터를 **눈으로** 확인
- **핵심 역량**: 문제를 말로 정확히 설명 → LLM 코드 생성 → Colab 실행/검증 → 에러는 다시 LLM 에게
- 다음 예제부터는 이 데이터로 **실제 예측 모델(회귀)** 을 만들어 봅니다.
